<a href="https://colab.research.google.com/github/Solo7602/web/blob/2lab/web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install requests pandas numpy matplotlib folium tqdm python-dateutil

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# Загружаем файл
df = pd.read_csv("hh_vacancies_clean.csv")

texts = df["text_clean"]  # текст вакансии

positive_kw = [
    "удаленн", "гибкий график", "современный офис"
]
negative_kw = [
    "стресс", "много задач", "большой объем", "жесткие сроки","срочная работа","жесткий контроль", "дисциплина", "давлен","высокая ответственность", "высокие требования"
]

def get_sentiment(text):
    text = str(text).lower()

    if any(w in text for w in positive_kw):
        return "positive"
    if any(w in text for w in negative_kw):
        return "negative"
    return "neutral"

df["sentiment"] = df["text_clean"].apply(get_sentiment)

# --- Классификация ---
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["sentiment"], test_size=0.2, random_state=42
)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = LogisticRegression(max_iter=300)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# --- Тематическое моделирование (LDA) ---
cv = CountVectorizer(max_features=5000, stop_words="english")
X_cv = cv.fit_transform(df["text_clean"])

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X_cv)

words = cv.get_feature_names_out()

print("\nТемы LDA:")
for i, topic in enumerate(lda.components_):
    top_words = [words[idx] for idx in topic.argsort()[-10:]]
    print(f"Тема {i+1}: {', '.join(top_words)}")

Accuracy: 0.75
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         1
     neutral       0.70      1.00      0.82        14
    positive       1.00      0.44      0.62         9

    accuracy                           0.75        24
   macro avg       0.57      0.48      0.48        24
weighted avg       0.78      0.75      0.71        24


Темы LDA:
Тема 1: alternate, по, на, working, response, accept, true, url, false, id
Тема 2: работы, для, accept, true, по, url, на, id, false, 1с
Тема 3: для, true, т1, по, опыт, на, url, работы, false, id
Тема 4: работы, на, для, от, магнит, true, url, по, false, id
Тема 5: от, мы, работы, анализ, по, url, для, на, id, false
